# glcuda Wave 130 - T4 device parity gate
Resource, module-load, occupancy, serial parity, and exact-output gate only. No production throughput claim.


In [ ]:
import base64
import gzip
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import traceback
import zipfile

BUILD = "wave130-device-parity-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
PATCH_SHA256 = "7fc9843b89aa443024d21ec7a4c3c9ce8b76f162aa03851dc8a0db3d778c3265"
PATCH_B64 = "..."
ROOT = Path("/kaggle/working/wave130")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
PATCH = ROOT / "wave130.patch"
CUBIN = ROOT / "wave130.cubin"
ARCHIVE = Path("/kaggle/working/glcuda-t4-wave130-device-parity-results.zip")
RESULTS.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, env=None, timeout=7200):
    cmd = [str(x) for x in cmd]
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    print("$", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, cwd=cwd, env=merged, text=True, capture_output=True, timeout=timeout)
    print(p.stdout[-30000:], flush=True)
    print(p.stderr[-30000:], flush=True)
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def save(name, process):
    (RESULTS / name).write_text(
        process.stdout + "\n--- STDERR ---\n" + process.stderr,
        encoding="utf-8",
    )


def archive():
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    print("ARCHIVE", ARCHIVE, hashlib.sha256(ARCHIVE.read_bytes()).hexdigest(), flush=True)


phase = "bootstrap"
try:
    patch = gzip.decompress(base64.b64decode(PATCH_B64))
    if hashlib.sha256(patch).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch SHA-256 mismatch")
    PATCH.write_bytes(patch)
    (RESULTS / "source.json").write_text(
        json.dumps(
            {
                "build": BUILD,
                "base_rev": BASE_REV,
                "patch_sha256": PATCH_SHA256,
                "patch_bytes": len(patch),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    phase = "gpu"
    smi = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    save("nvidia-smi.log", smi)
    if "Tesla T4" not in smi.stdout:
        raise RuntimeError("Wave 130 requires a Tesla T4")

    phase = "checkout"
    clone = run(["git", "clone", REPO_URL, TREE], cwd=ROOT)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE)
    save("git-checkout.log", checkout)
    apply_check = run(["git", "apply", "--check", PATCH], cwd=TREE)
    save("git-apply-check.log", apply_check)
    applied = run(["git", "apply", PATCH], cwd=TREE)
    save("git-apply.log", applied)
    diff_check = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff_check)

    cargo = shutil.which("cargo")
    common = {"CUDA_VISIBLE_DEVICES": "0", "CARGO_TARGET_DIR": str(TARGET)}
    if cargo is None:
        phase = "rustup"
        install = run(
            [
                "bash",
                "-lc",
                "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
            ],
            timeout=1200,
        )
        save("rustup-install.log", install)
        cargo = str(Path.home() / ".cargo/bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable: {cargo}")

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "68 passed; 0 failed" not in tests.stdout:
        raise RuntimeError("host test contract changed")

    phase = "compiler-resource"
    ptxas = Path("/usr/local/cuda/bin/ptxas")
    if not ptxas.is_file():
        raise RuntimeError(f"ptxas unavailable: {ptxas}")
    built = run(
        [ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75_wave129.ptx", "-o", CUBIN]
    )
    save("ptxas-wave130.log", built)
    log = built.stdout + "\n" + built.stderr

    def amount(pattern):
        found = re.search(pattern, log)
        if not found:
            raise RuntimeError(f"missing ptxas field: {pattern}")
        return int(found.group(1))

    resource = {
        "registers_per_thread": amount(r"Used\s+(\d+)\s+registers"),
        "barriers": amount(r"used\s+(\d+)\s+barriers"),
        "static_shared_bytes": amount(r"(\d+)\s+bytes smem"),
        "stack_frame_bytes": amount(r"(\d+)\s+bytes stack frame"),
        "spill_store_bytes": amount(r"(\d+)\s+bytes spill stores"),
        "spill_load_bytes": amount(r"(\d+)\s+bytes spill loads"),
    }
    (RESULTS / "resource.json").write_text(json.dumps(resource, indent=2), encoding="utf-8")
    if resource != {
        "registers_per_thread": 80,
        "barriers": 1,
        "static_shared_bytes": 16384,
        "stack_frame_bytes": 0,
        "spill_store_bytes": 0,
        "spill_load_bytes": 0,
    }:
        raise RuntimeError(f"Wave 130 resource gate failed: {resource}")

    candidate_env = {
        **common,
        "GLCUDA_FORCE_Q8": "1",
        "GLCUDA_GRID2D": "1",
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_Q8_NOSTORE": "1",
        "GLCUDA_FFN_GATE_UP_STACKED": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_GEMM_N16_PREFETCH": "1",
        "GLCUDA_N16_FUSED_SWIGLU": "1",
        "RUST_BACKTRACE": "1",
    }

    phase = "build"
    build = run(
        [cargo, "build", "--release", "-p", "glcuda", "--example", "wave130_n16_fused_swiglu", "--locked"],
        cwd=TREE,
        env=candidate_env,
    )
    save("cargo-build.log", build)

    phase = "cuda-parity"
    parity = run(
        [cargo, "test", "--release", "-p", "glcuda", "--test", "parity", "--locked", "--", "--nocapture", "--test-threads=1"],
        cwd=TREE,
        env=candidate_env,
    )
    save("cargo-cuda-parity.log", parity)

    phase = "direct"
    direct = run([TARGET / "release/examples/wave130_n16_fused_swiglu"], cwd=TREE, env=candidate_env)
    save("wave130-direct.log", direct)
    direct_match = re.search(r"\[wave130-direct\]\s*(\{[^\n]+\})", direct.stdout)
    occupancy_match = re.search(r"\[wave130-resource\]\s*(\{[^\n]+\})", direct.stdout)
    if not direct_match or not occupancy_match:
        raise RuntimeError("Wave 130 direct records missing")
    direct_record = json.loads(direct_match.group(1))
    occupancy_record = json.loads(occupancy_match.group(1))
    if occupancy_record["active_blocks_per_sm"] < 3:
        raise RuntimeError(f"occupancy gate failed: {occupancy_record}")
    if not direct_record["q8_bit_exact"] or not direct_record["scale_bit_exact"]:
        raise RuntimeError(f"exact output gate failed: {direct_record}")
    if direct_record["full_slabs"] != 3 or direct_record["ragged_tail_rows"] != 52:
        raise RuntimeError(f"production tail contract failed: {direct_record}")

    summary = {
        "build": BUILD,
        "gpu": "Tesla T4",
        "host_tests": {"passed": 68, "failed": 0},
        "resource": resource,
        "cuda_parity_passed": True,
        "occupancy": occupancy_record,
        "direct": direct_record,
        "production_runner_wired": False,
        "production_timing_run": False,
        "target_15000_tps_achieved": False,
    }
    (RESULTS / "wave130-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE130_RESULT", json.dumps(summary, indent=2), flush=True)
except Exception:
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": traceback.format_exc()}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise

archive()